In [55]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import FileReadTool, ScrapeWebsiteTool, CSVSearchTool, SerperDevTool, \
    EXASearchTool, CodeInterpreterTool
from langchain_openai import ChatOpenAI
import os
from exa_py import Exa
from utils import get_openai_api_key, get_serper_api_key, get_exa_api_key

In [56]:
# Get api keys
openai_api_key = get_openai_api_key()
exa_api_key = get_exa_api_key()
serper_api_key = get_serper_api_key()

In [ ]:
serper_search = SerperDevTool()
exa_search = EXASearchTool()
scrape_tool = ScrapeWebsiteTool()
def_read_tool = FileReadTool(file_path='project_definition.txt')

In [ ]:
# Agent: Project Supervisor
project_supervisor = Agent(
    role = "Project Supervisor",
    goal = """Your goal is to"
        1. direct, review, guide and organize the academic research project.\
        2. Ensure a comprehensive, logical and coherent structure \
        3. Ensure relevant statistical analysis are provided for the research project that demonstrate\
            expertise and deep knowledge in statistical analysis techniques and statistical computing. \
        4. contents are relevant to the domain of available data""", 
    backstory = """You are a Professor of Statistics supervising an academic research project. You maintain \
        oversight of the project to meet the quality, content and formatting standards for the award of \
        Post Graduate Diploma in Statistics at the department of statistics in the faculty of science of a \
        leading, world-class, University. You do not allow any form of plagiarism""",
    allow_delegation=True,
	verbose=True
)


In [ ]:
# Agent: Public Health Specialist
malaria_specialist = Agent(
    role = "Malaria Specialist",
    goal = """Your goal is to ensure. \
        1. insightful, factual and accurate contents about Malaria to the research \
        2. cotents follow a logical and coherent structure that allign to industry standards for \
            target audience""",
    backstory = "You are a Public health professional specializing in Malaria in Africa. \
        You understand the dynamic of the desease, transmission, its impact on Africa and \
        initiatives to combat the desease and their effectiveness.",
    allow_delegation=True,
	verbose=True
)


In [ ]:
#Agent: Content Planner
content_planner = Agent(
    role = "Content Planner",
    goal = "Plan robust, coherent, engaging and comprehensive academic report content outlines",
    backstory = "You are planning an academic research report content on project topic"
        "You collaborate with Public Health Specialist and Project Supervisor to make informed decisions."
        "on the structure and outlines for the contents"
        "Your work is the basis for Content Researcher and the Content Writer.",
    allow_delegation=False,
    tools = [def_read_tool],
	verbose=True
)

In [61]:
# Agent: Content Researcher
content_researcher = Agent(
    role = """Content Researcher""",
    goal = """Your goal is to: \
        1. research credible sources of insightful, accurate and factual information \
            including websites and publications from WHO, UNICEF, CDC, and other domain experts in \
            West Africa and specifically those in the selected countries in the dataset \
        2. collect and reference insightful, accurate and factual information from the \
            sources relevant to the project domain.""",
    backstory = """You are a seasoned, resourceful and prolific content researcher for academic writing \
        You find the most relevant accurate and factual information \
        Your work is the basis for the work of Content Writer""",
    allow_delegation = False,
    tools = [exa_search],
    verbose = True
)


In [ ]:
# Agent: Data Analyst
data_analyst = Agent(
    role = "Data Analyst",
    goal = "Your goal is to conduct statistical data analysis required and generate insightful visualization",
    backstory = "You generate R code for statistical analysis and data visualization that communicate \
        insights in accordance with the relevant statistical analysis required or as directed by Project Supervisor",
    allow_delegation = False,
	verbose=True
)


In [ ]:
# Agent: Content Writer
content_writer = Agent(
    role = "Content Writer",
    goal = "Write robust, comprehensive, accurate and factual content for academic research report",
    backstory = """
        1. You use the results from Content Researcher and Data Analyst to write clear, \
        concise and engaging academic research contents. \
        2. You avoid plagiarisim and use inline citation and referencing of sources. \
        3. Include R code for statistical analysis and visualization from Data Analyst\
        """,
    allow_delegation=False,
	verbose=True
)

In [ ]:
plan = Task(
    description = (
        "1. determine logical structure for the contents for {chapter} in academic research report on {project}"
        "2. create an organized content outlines"
        "3. review and finalize the content outlines"
    ),
    expected_output = (
        "A content plan that provides a logical and engaging structure of content outlines"
    ),
    tools = [exa_search, scrape_tool, def_read_tool],
    agent = content_planner

)

In [ ]:
research = Task(
    description = (
        "1. Research credible sources of insightful, accurate and factual contents over the internet for \
            {chapter} in academic research report on {project}"
        "2. furnish relevant content requirements for the research project report as indicated in content plan"
        "3. respond to relevant review reports as required"
    ),
    expected_output = (
        "A research report providing contents with attributions to sources as required in academic research"
    ),
    tools = [exa_search, scrape_tool, def_read_tool],
    agent = content_researcher

)

In [ ]:
# Task
write = Task(
    description = (
        "1. Write robust, comprehensive, factual, accurate and engaginging content based on the \
            research report and content plan for {chapter} in academic research report on {project}"
        "2. collaborate with Project Supervisor, Malaria Specialist, Data Analyst and Content Researcher \
            and Content Planner"
        "3. resolve gramatical and typographical errors"
        "4. give credit to sources of ideas you use"
    ),
    expected_output="""Detailed, robust, comprehensive, insightful and engaging academic research content\
        with additional section for references""",
    tools = [exa_search, scrape_tool, def_read_tool],
    agent=content_writer
)

In [ ]:
specialist_review = Task(
    description = "Review contents provided for {chapter} in academic research report on {project} \
        and direct actions to ensure:"
        "1. factual, accurate and quality conent for accademic research"
        "2. professional communication relevant to the industry"
        "3. robust and coherently organized content with a consistent style and engaging presentation"
        "4. contribute to content outlines structure, style and logic"
        "5. no plagiarism",
    expected_output = "specialist review report",
    tools = [exa_search, scrape_tool, def_read_tool],
    agent = malaria_specialist
)

In [68]:
supervisor_review = Task(
    description = """" Review contents provided {chapter} in academic research report on {project} and \
        direct actions to:
        1. Ensure content structure is logical, coherent and address relevant aspects of the research. \
        2. Ensure contents are robust and comprehensive. \
        3. Ensure resources are from credible sources and subject matter experts in the research domain. \
        4. Ensure associated references and citations are provided as required for academic research. \
        5. R code for relevant statistical analysis and visualizations required are provided. \
        6. There is no plagiarism, ideas and citations are appropriately given credit and referenced.""",
    expected_output = "Supervisor review report",
    tools = [scrape_tool],
    agent=project_supervisor
)

In [ ]:
# Crew
crew = Crew(
    agents=[content_planner, content_researcher, content_writer,  malaria_specialist], 
    tasks=[plan, research, specialist_review, supervisor_review, write],
    manager_agent=project_supervisor,
    process=Process.hierarchical,
    verbose=True, 
    memory=True
)


2024-12-09 23:55:48,373 - 12852 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [71]:
result = crew.kickoff(
    inputs={
        "chapter": "CHAPTER ONE: Introduction",
        "project": "Analysis of Malaria Incidence Rates in Five West African Countries"
    }
)

# Agent: Project Supervisor
## Task: 1. determine logical structure for the contents for CHAPTER ONE: Introduction in academic research report on Analysis of Malaria Incidence Rates in Five West African Countries2. create an organized content outlines3. review and finalize the content outlines
 

I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 2 validation errors for DelegateWorkToolSchema
task
  Input should be a valid string [type=string_type, input_value={'description': 'Develop ...resses.', 'type': 'str'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type
context
  Input should be a valid string [type=string_type, input_value={'description': 'The acad...veness.', 'type': 'str'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type.
 Tool Delegate work to coworker accepts these inputs: Tool Name: Delegate work to coworker
Tool Arguments: 

In [72]:
result

CrewOutput(raw="### Chapter One: Introduction\n\n#### Background on Malaria\n\nMalaria is a mosquito-borne infectious disease caused by parasitic protozoans of the genus Plasmodium, primarily transmitted through the bite of infected female Anopheles mosquitoes. This disease remains a significant global health challenge, especially in tropical and subtropical regions. The World Health Organization (WHO) reported that in 2020, there were an estimated 241 million cases of malaria worldwide, with the African Region bearing the highest burden, accounting for 95% of cases and deaths. The clinical manifestations of malaria can vary widely, ranging from asymptomatic infections to severe illness characterized by fever, chills, and flu-like symptoms. In severe cases, malaria can lead to complications such as cerebral malaria, anemia, and multi-organ failure.\n\nHistorically, malaria has shaped populations and economies across continents. The colonial period saw extensive research into malaria’s 

In [73]:
print(result)

### Chapter One: Introduction

#### Background on Malaria

Malaria is a mosquito-borne infectious disease caused by parasitic protozoans of the genus Plasmodium, primarily transmitted through the bite of infected female Anopheles mosquitoes. This disease remains a significant global health challenge, especially in tropical and subtropical regions. The World Health Organization (WHO) reported that in 2020, there were an estimated 241 million cases of malaria worldwide, with the African Region bearing the highest burden, accounting for 95% of cases and deaths. The clinical manifestations of malaria can vary widely, ranging from asymptomatic infections to severe illness characterized by fever, chills, and flu-like symptoms. In severe cases, malaria can lead to complications such as cerebral malaria, anemia, and multi-organ failure.

Historically, malaria has shaped populations and economies across continents. The colonial period saw extensive research into malaria’s transmission dynamics 